# Inspecting `subsample.parquet`

This notebook is a reference: open it any time to remember what the file looks like.

**What we look at:**
1. The whole file at a glance (all 48 columns).
2. The 5 columns we actually load.
3. One document, field by field — exactly what BM25/RAG see.
4. How much HTML is in each text field.
5. What the HTML looks like and what stripping removes.

In [ ]:
import sys
from pathlib import Path

# Make `ows_retrieval` importable when running the notebook from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import html
import re

import pandas as pd
import pyarrow.parquet as pq

from ows_retrieval.data import DATA_RAW, load_docs
from ows_retrieval.bm25 import doc_text

pd.set_option("display.max_colwidth", 120)

## 1. The whole file at a glance

The file has 48 columns, but only a few are text. Note there is **no clean-text / `default_text` column** — the only document body is `main_content`, and it contains raw HTML.

In [ ]:
schema = pq.read_schema(DATA_RAW / "subsample.parquet")
print(f"{len(schema.names)} columns, {pq.read_metadata(DATA_RAW / 'subsample.parquet').num_rows} rows\n")
for name, dtype in zip(schema.names, schema.types):
    print(f"  {name:60s} {dtype}")

## 2. The columns we actually use

`load_docs()` reads only these 5: `id, title, description, main_content, url`. `id` is the SHA-256 string that matches `doc_id` in qrels.

In [ ]:
docs = load_docs()
print("rows (documents):", len(docs))
print("columns:", docs.columns.tolist())
print("non-null counts:")
print(docs.notna().sum())
docs.head(3)

## 3. One document, field by field

This is exactly what BM25 indexes and what the RAG context feeds the LLM. `doc_text()` concatenates `title + description + main_content` (skipping NaNs).

In [ ]:
row = docs.iloc[0]
for field in ["id", "url", "title", "description", "main_content"]:
    value = row[field]
    shown = "<NaN>" if pd.isna(value) else repr(str(value)[:300])
    print(f"--- {field} ---")
    print(shown)
    print()

print("=== doc_text() (what BM25 tokenizes / RAG shows) ===")
print(repr(doc_text(row)[:400]))

## 4. How much HTML is in each text field?

We count the fraction of documents whose field contains an HTML tag (`<...>`) or an HTML entity (`&nbsp;`, `&amp;`, `&#39;`, ...).

In [ ]:
TAG = re.compile(r"<[a-zA-Z/][^>]*>")
ENTITY = re.compile(r"&[a-zA-Z]+;|&#\d+;")

for col in ["title", "description", "main_content"]:
    s = docs[col].dropna().astype(str)
    print(f"{col:14s} n={len(s):6d}  frac_with_tag={s.str.contains(TAG).mean():.3f}  "
          f"frac_with_entity={s.str.contains(ENTITY).mean():.3f}")

## 5. What the HTML looks like, and what stripping removes

`strip_html` below removes tags and unescapes entities — a rough version of what the dense corpus already had done to it. We show one document before/after and the corpus-wide size reduction (≈ 28% of BM25 tokens are HTML-derived).

In [ ]:
WS = re.compile(r"\s+")

def strip_html(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    return WS.sub(" ", text).strip()

# find a doc whose main_content actually has tags
mc = docs["main_content"].dropna().astype(str)
sample = mc[mc.str.contains(TAG)].iloc[0]

print("--- BEFORE (raw main_content) ---")
print(repr(sample[:400]))
print("\n--- AFTER strip_html ---")
print(repr(strip_html(sample)[:400]))

In [ ]:
# Character reduction from stripping HTML, estimated on a 2,000-doc sample.
# (The full corpus is ~1 GB of main_content text, so the whole-corpus version
# takes ~40s; the sample gives the same percentage in ~1s.)
sample_mc = mc.sample(2000, random_state=0)
raw_len = sample_mc.str.len()
clean_len = sample_mc.map(lambda t: len(strip_html(t)))
reduction = 1 - clean_len.sum() / raw_len.sum()
print(f"median raw main_content chars     : {int(raw_len.median())}")
print(f"median stripped main_content chars: {int(clean_len.median())}")
print(f"characters removed by stripping (2k sample): {reduction:.1%}")

## Summary — what this notebook showed

| Property | Value |
|---|---|
| rows (documents) | ~63,639 |
| columns | 48 (only 5 loaded by `load_docs`) |
| body text field | `main_content` (the only one) |
| HTML in `main_content` | ~97% have tags, ~55% have entities |
| clean-text column | **none** — must strip HTML yourself |